# 02 - Modelos clásicos de ML para predecir la dirección del día siguiente

**Objetivo:** probar con datos reales qué familia de modelos clásicos de `scikit-learn` le gana al baseline, usando el mismo esquema que en el Módulo 4 del diplomado anterior (`model_params` + `GridSearchCV`, ROC con umbral de Youden, importancia de variables), pero **adaptado a series de tiempo**.

**Qué es igual que en el Módulo 4:** regresión logística (l1/l2), árboles con poda, KNN, SVM (kernels linear/poly/rbf/sigmoid), bagging y boosting, normalización, validación cruzada, curva ROC.

**Qué cambia (y por qué; esto se defiende en la exposición):**

| Módulo 4 (banco) | Aquí (acciones) | Motivo |
|---|---|---|
| `train_test_split` aleatorio 80/20 | Corte por **fecha**; últimos 6 meses de *holdout* | Con filas mezcladas, el modelo vería el futuro al entrenar. |
| `cv=10` (KFold) | `TimeSeriesSplit` con `gap`, cortando por **fechas** | Cada pliegue valida siempre en fechas posteriores a las de entrenamiento. |
| Escalar todo y luego dividir | `StandardScaler` **dentro** del `Pipeline` | Así se ajusta solo con el entrenamiento de cada pliegue (sin fuga). |
| Métrica: recall (clase rara) | **AUC-ROC** (principal) y **F1** | Las clases aquí están casi balanceadas; el documento pide F1 y AUC. |
| SMOTE para desbalance | No se usa (ver sección 5) | Datos sintéticos entre días vecinos rompen la estructura temporal. |
| Umbral de Youden sobre *test* | Umbral de Youden sobre predicciones **fuera de muestra de la validación** | El *holdout* no se toca para elegir nada. |

> Es un notebook de **exploración**: lo que funcione aquí se moverá después a funciones en `src/`. Los números reales dependen de la fecha de descarga; no hay resultados escritos en el texto a propósito.

## 1. Parámetros desde `configs/config.yaml`

Nada de valores fijos en el notebook: tickers, ventanas y parámetros de validación se leen del YAML (el mismo archivo que leerá `src/` más adelante).

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
import yfinance as yf
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    TimeSeriesSplit,
    cross_validate,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
with open(RAIZ / "configs" / "config.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

OBJETIVO = cfg["datos"]["tickers_objetivo"]
CONTEXTO = cfg["datos"]["tickers_contexto"]
INICIO = cfg["datos"]["fecha_inicio"]
H = cfg["datos"]["horizonte_dias"]
VENT_RET = cfg["features"]["ventanas_retorno"]
VENT_ROLL = cfg["features"]["ventanas_rolling"]
VENT_CTX = cfg["features"]["ventanas_contexto"]
RSI_N = cfg["features"]["rsi_periodo"]
N_SPLITS = cfg["validacion"]["n_splits"]
GAP = cfg["validacion"]["gap_dias"]
MESES_HOLDOUT = cfg["validacion"]["meses_holdout"]
SEMILLA = cfg["validacion"]["semilla"]

print("Objetivo:", OBJETIVO)
print("Contexto:", CONTEXTO)

## 2. Datos

Se usa la caché que dejó el notebook 01 (`data/exploracion/`, que Git ignora). Si no existe, se descarga. Para forzar una descarga fresca, borra esa carpeta.

In [ ]:
def cargar_precios(tickers):
    carpeta = RAIZ / "data" / "exploracion"
    f_c, f_v = carpeta / "close.csv", carpeta / "volume.csv"
    if f_c.exists() and f_v.exists():
        c = pd.read_csv(f_c, index_col=0, parse_dates=True)
        v = pd.read_csv(f_v, index_col=0, parse_dates=True)
        if set(tickers) <= set(c.columns):
            return c[tickers], v[tickers]
    datos = yf.download(tickers, start=INICIO, auto_adjust=True, progress=False)
    carpeta.mkdir(parents=True, exist_ok=True)
    datos["Close"].to_csv(f_c)
    datos["Volume"].to_csv(f_v)
    return datos["Close"][tickers], datos["Volume"][tickers]


cierre, volumen = cargar_precios(OBJETIVO + CONTEXTO)
print(cierre.shape, cierre.index.min().date(), "->", cierre.index.max().date())
cierre.isna().sum()

## 3. Features y etiqueta

Reglas que no se rompen:

- **Solo pasado en las características.** Todo se calcula con datos hasta el cierre del día *t*.
- **La etiqueta mira un día al futuro:** `y = 1` si el cierre de *t+1* es mayor que el de *t*. El último día de cada serie no tiene futuro conocido y **se elimina** (no se etiqueta como "baja": comparar contra `NaN` da `False` y metería una etiqueta falsa).
- **Retornos, no precios:** el precio sube con el tiempo y no generaliza entre acciones. Las características no incluyen el nombre del ticker, así el modelo es el mismo para todas.

Familias de características: retornos acumulados de 1, 2, 5 y 10 días, media y desviación móvil del retorno diario (5, 10 y 20 días), RSI, distancia a la media móvil de 20 días y volumen relativo (volumen del día entre su media de 20 días).

Características de **contexto**: retorno de SPY y NVDA en 1 y 5 días hasta el día *t* (mismas para todos los tickers ese día). Se prueban aparte para medir si ayudan.

In [ ]:
def rsi(cierre_t, n):
    delta = cierre_t.diff()
    kw = {"alpha": 1 / n, "adjust": False, "min_periods": n}
    ganancia = delta.clip(lower=0).ewm(**kw).mean()
    perdida = (-delta.clip(upper=0)).ewm(**kw).mean()
    return 100 - 100 / (1 + ganancia / perdida)


def features_propias(c, v):
    ret = c.pct_change(fill_method=None)
    f = pd.DataFrame(index=c.index)
    for k in VENT_RET:
        f[f"ret_{k}d"] = c.pct_change(k, fill_method=None)
    for k in VENT_ROLL:
        f[f"media_{k}d"] = ret.rolling(k).mean()
        f[f"desv_{k}d"] = ret.rolling(k).std()
    f[f"rsi_{RSI_N}"] = rsi(c, RSI_N)
    f[f"dist_media_{VENT_ROLL[-1]}d"] = c / c.rolling(VENT_ROLL[-1]).mean() - 1
    f["vol_rel"] = v / v.rolling(VENT_ROLL[-1]).mean()
    return f


def features_contexto(cierre_ctx):
    f = pd.DataFrame(index=cierre_ctx.index)
    for t in cierre_ctx.columns:
        for k in VENT_CTX:
            f[f"ctx_{t}_ret_{k}d"] = cierre_ctx[t].pct_change(k, fill_method=None)
    return f


F_CTX = features_contexto(cierre[CONTEXTO])

paneles = []
for t in OBJETIVO:
    c = cierre[t].dropna()
    v = volumen[t].reindex(c.index)
    f = features_propias(c, v).join(F_CTX)
    futuro = c.shift(-H)
    f["y"] = (futuro > c).astype(int)
    f = f[futuro.notna()]
    f["ticker"] = t
    paneles.append(f.rename_axis("fecha"))

panel = pd.concat(paneles).reset_index()
panel = panel.replace([np.inf, -np.inf], np.nan).dropna()
panel = panel.sort_values(["fecha", "ticker"]).reset_index(drop=True)

FECHA_CORTE = pd.Timestamp(cfg["validacion"]["fecha_corte"])
adelante = panel[panel["fecha"] > FECHA_CORTE].reset_index(drop=True)
panel = panel[panel["fecha"] <= FECHA_CORTE].reset_index(drop=True)
print(
    "Fecha de corte:",
    FECHA_CORTE.date(),
    f"| filas posteriores (prueba hacia adelante): {len(adelante):,}",
)

COLS_CTX = [c for c in panel.columns if c.startswith("ctx_")]
NO_FEATURE = ["fecha", "ticker", "y", *COLS_CTX]
COLS_PROPIAS = [c for c in panel.columns if c not in NO_FEATURE]
print("Filas:", len(panel), "| tickers:", panel["ticker"].nunique())
print("Propias:", COLS_PROPIAS)
print("Contexto:", COLS_CTX)

**Prueba anti-fuga.** Para un día cualquiera, la característica `ret_1d` debe coincidir con el cambio de precio entre *t-1* y *t*, y la etiqueta con la comparación entre *t* y *t+1*. Si esta celda falla, algo está mal en el cálculo.

In [ ]:
t0 = OBJETIVO[0]
c0 = cierre[t0].dropna()
d0 = panel.loc[panel["ticker"] == t0, "fecha"].iloc[100]
i0 = c0.index.get_loc(d0)
fila = panel[(panel["ticker"] == t0) & (panel["fecha"] == d0)].iloc[0]

assert np.isclose(fila["ret_1d"], c0.iloc[i0] / c0.iloc[i0 - 1] - 1)
assert fila["y"] == int(c0.iloc[i0 + 1] > c0.iloc[i0])
print("OK:", t0, d0.date(), "- ret_1d usa t-1 y t; y usa t y t+1")

## 4. Partición temporal: entrenamiento y *holdout*

Los últimos `meses_holdout` meses se reservan y **no se usan para elegir nada**. Entre el final del entrenamiento y el inicio del holdout se deja un hueco de `gap_dias` días hábiles. La partición es por **fecha** (no por ticker): todas las acciones de un mismo día caen del mismo lado.

Los datos **posteriores a `fecha_corte`** (septiembre de 2026 en el YAML) no entran a ninguna etapa: quedan aparte como *prueba hacia adelante* para la app. Los 6 meses de holdout se cuentan hacia atrás desde esa fecha de corte.

In [ ]:
fechas = np.sort(panel["fecha"].unique())
corte = pd.Timestamp(fechas[-1]) - pd.DateOffset(months=MESES_HOLDOUT)
i_hold = int(np.searchsorted(fechas, np.datetime64(corte)))
fin_train = fechas[i_hold - GAP - 1]

train = panel[panel["fecha"] <= fin_train].reset_index(drop=True)
hold = panel[panel["fecha"] >= fechas[i_hold]].reset_index(drop=True)

print(
    "Train  :",
    train["fecha"].min().date(),
    "->",
    train["fecha"].max().date(),
    f"({len(train):,} filas)",
)
print(
    "Holdout:",
    hold["fecha"].min().date(),
    "->",
    hold["fecha"].max().date(),
    f"({len(hold):,} filas)",
)

X_train, y_train = train[COLS_PROPIAS], train["y"]
X_train_ctx = train[COLS_PROPIAS + COLS_CTX]

In [ ]:
dias_al_alza = train.groupby("ticker")["y"].mean().sort_values() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(dias_al_alza.index, dias_al_alza.values, color="tab:blue")
ax.axvline(50, color="k", linestyle="--", linewidth=1, label="50 %")
ax.set_title("Porcentaje de días al alza por ticker (entrenamiento)")
ax.set_xlabel("% de días con cierre siguiente mayor")
ax.set_ylabel("Ticker")
ax.legend()
plt.show()

print(f"Global: {y_train.mean():.1%} de los días suben")

## 5. Desbalance: ¿hace falta SMOTE?

En el Módulo 4 la clase "contrata" era rara y se usó SMOTE. Aquí las clases están casi balanceadas (ver porcentaje de arriba), así que el problema no es el desbalance. Además, SMOTE crea puntos nuevos interpolando entre vecinos, y en series de tiempo esos puntos mezclan días distintos y fabrican patrones que no existieron. Lo que sí probamos, dentro de la regresión logística, es `class_weight="balanced"`: no inventa datos, solo pondera.

**Consecuencia para la métrica.** Si siempre se responde "sube", el F1 de la clase "sube" ya es alto (si suben el 53 % de los días, ronda 0.69), porque casi nunca se equivoca en esa clase. Por eso el baseline en F1 es difícil de superar y la métrica principal es el **AUC-ROC**: mide si el modelo *ordena* mejor los días buenos y malos, y el baseline obtiene 0.50 exacto.

## 6. Validación cruzada temporal (por fechas)

`TimeSeriesSplit` sobre las **fechas únicas** del entrenamiento, con ventana creciente: el pliegue 1 entrena con lo más antiguo y valida en el bloque siguiente; el pliegue 2 entrena con lo anterior más ese bloque y valida en el siguiente; y así. El `gap` separa entrenamiento y validación. La función devuelve índices de **filas**, que `GridSearchCV` acepta directamente.

In [ ]:
def splits_por_fecha(fechas_fila, n_splits, gap):
    unicas = np.sort(pd.unique(fechas_fila))
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    salida = []
    for i_tr, i_va in tscv.split(unicas):
        tr = np.flatnonzero(np.isin(fechas_fila, unicas[i_tr]))
        va = np.flatnonzero(np.isin(fechas_fila, unicas[i_va]))
        salida.append((tr, va))
    return salida


CV = splits_por_fecha(train["fecha"].to_numpy(), N_SPLITS, GAP)

resumen_cv = pd.DataFrame(
    [
        {
            "pliegue": i + 1,
            "train_hasta": train["fecha"].iloc[tr].max().date(),
            "valida_desde": train["fecha"].iloc[va].min().date(),
            "valida_hasta": train["fecha"].iloc[va].max().date(),
            "filas_train": len(tr),
            "filas_valida": len(va),
        }
        for i, (tr, va) in enumerate(CV)
    ]
)
resumen_cv

## 7. Baseline y tamizado de modelos (estilo `model_params` del Módulo 4)

Cada modelo se afina con una búsqueda de hiperparámetros sobre los mismos pliegues temporales. Se calculan **AUC y F1** y se elige el mejor por AUC (`refit="auc"`).

**Dos tipos de búsqueda** (el documento acepta ambas):

- `GridSearchCV` prueba **todas** las combinaciones. Se usa en los modelos con espacios chicos o rápidos: regresión logística, KNN, árbol, bagging y SVM.
- `RandomizedSearchCV` prueba un número fijo de combinaciones **al azar** (`n_iter`) dentro de un espacio grande. Se usa en bosque, AdaBoost y boosting, que tienen muchos hiperparámetros y son lentos: con el mismo presupuesto explora más hiperparámetros distintos que una rejilla completa (idea de Bergstra y Bengio, 2012).

Después de la tabla hay un **mapa de lo explorado** (qué se probó) y una columna `en_borde` que avisa si el mejor valor quedó en el extremo de su lista, señal de que el óptimo puede estar más allá. Todo el notebook tarda unos 10 a 15 minutos.

Detalles que importan (y que conviene saber explicar):

- Regresión logística, KNN y SVM llevan escalador **dentro** del `Pipeline`. En la logística se compara `StandardScaler` contra `RobustScaler` (usa mediana y rango intercuartil, menos sensible a los retornos extremos). Los árboles no necesitan escalar.
- `HistGradientBoostingClassifier` trae por defecto un *early stopping* que, con más de 10,000 filas, separa un trozo **al azar** del entrenamiento para validar. En series de tiempo eso mezcla el orden, así que se desactiva con `early_stopping=False`.
- KNN con muchos vecinos (50 a 400) porque el ruido de los retornos diarios es alto; con pocos vecinos memoriza ruido.
- `auc_train` junto a `auc_cv` sirve para detectar sobreajuste, igual que en el Módulo 4: si el de entrenamiento es mucho mayor, el modelo memoriza. En KNN no se calcula porque es lento.
- Regresión logística: desde scikit-learn 1.8 el parámetro `penalty` está obsoleto. Se usa `l1_ratio` (0 equivale a l2/Ridge y 1 a l1/Lasso).
- **Cuidado con buscar de más:** cuantas más combinaciones se prueban, más probable es que el ganador lo sea por suerte en la validación, así que su AUC queda algo optimista. Por eso existe el *holdout* que no se toca.

In [ ]:
SCORING = {"f1": "f1", "auc": "roc_auc"}
m_cfg = cfg["modelos"]
ESCALADORES = [StandardScaler(), RobustScaler()]

model_params = {
    "dummy": {
        "model": DummyClassifier(strategy="most_frequent"),
        "params": {"strategy": ["most_frequent"]},
    },
    "logistica": {
        "model": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(solver="liblinear", max_iter=1000)),
            ]
        ),
        "params": {
            "scaler": ESCALADORES,
            "clf__l1_ratio": [0, 1],  # 0 = l2 (Ridge), 1 = l1 (Lasso)
            "clf__C": m_cfg["logistica"]["C"],
            "clf__class_weight": [None, "balanced"],
        },
    },
    "knn": {
        "model": Pipeline(
            [("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]
        ),
        "params": {
            "clf__n_neighbors": [50, 100, 200, 400],
            "clf__weights": ["uniform", "distance"],
            "clf__p": [1, 2],  # 1 = distancia Manhattan, 2 = Euclidiana
        },
        "train": False,  # KNN compara contra todo el entrenamiento: es lento
    },
    "arbol": {
        "model": DecisionTreeClassifier(random_state=SEMILLA),
        "params": {
            "max_depth": [2, 3, 4, 5, 7],
            "min_samples_leaf": [25, 50, 100, 200],
            "criterion": ["gini", "entropy"],
            "ccp_alpha": [0.0, 0.0005, 0.002],  # poda por costo-complejidad
        },
    },
    "bagging": {
        "model": BaggingClassifier(
            DecisionTreeClassifier(min_samples_leaf=50),
            n_estimators=100,
            random_state=SEMILLA,
        ),
        "params": {
            "estimator__max_depth": [2, 3, 5],
            "max_samples": [0.3, 0.6],
            "max_features": [0.5, 1.0],
        },
    },
    "bosque": {
        "model": RandomForestClassifier(n_estimators=200, random_state=SEMILLA),
        "params": {
            "max_depth": [3, 4, 5, 6, 8],
            "min_samples_leaf": [25, 50, 100, 200],
            "max_features": ["sqrt", 0.3, 0.5],
            "max_samples": [0.3, 0.5, 0.8],
            "criterion": ["gini", "entropy"],
            "ccp_alpha": [0.0, 0.0005, 0.001],
            "class_weight": [None, "balanced_subsample"],
        },
        "n_iter": 16,
    },
    "adaboost": {
        "model": AdaBoostClassifier(
            DecisionTreeClassifier(max_depth=1), random_state=SEMILLA
        ),
        "params": {
            "estimator__max_depth": [1, 2, 3],
            "n_estimators": [50, 100, 200],
            "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
        },
        "n_iter": 10,
    },
    "boosting": {
        "model": HistGradientBoostingClassifier(
            early_stopping=False, random_state=SEMILLA
        ),
        "params": {
            "max_depth": m_cfg["boosting"]["max_depth"],
            "learning_rate": m_cfg["boosting"]["learning_rate"],
            "max_iter": [100, 200, 300],
            "min_samples_leaf": [50, 100, 200],
            "l2_regularization": [0.0, 1.0, 10.0],
            "max_leaf_nodes": [8, 15, 31],
        },
        "n_iter": 25,
    },
}


def en_borde(espacio, mejores):
    borde = []
    for k, valores in espacio.items():
        v = mejores[k]
        es_num = [
            isinstance(x, int | float) and not isinstance(x, bool) for x in valores
        ]
        if all(es_num) and len(valores) > 2 and v in (min(valores), max(valores)):
            borde.append(k.split("__")[-1])
    return borde


def espacio_explorado(model_params):
    filas = []
    for modelo, mp in model_params.items():
        for k, valores in mp["params"].items():
            etiquetas = [type(v).__name__ if hasattr(v, "fit") else v for v in valores]
            filas.append(
                {
                    "modelo": modelo,
                    "hiperparametro": k.split("__")[-1],
                    "valores": etiquetas,
                }
            )
    return pd.DataFrame(filas)


def tamizar(model_params, X, y, cv):
    filas, ganadores = [], {}
    for nombre, mp in model_params.items():
        t_ini = time.perf_counter()
        con_train = mp.get("train", True)
        comunes = {
            "cv": cv,
            "scoring": SCORING,
            "refit": "auc",
            "n_jobs": -1,
            "return_train_score": con_train,
        }
        if "n_iter" in mp:
            busqueda = RandomizedSearchCV(
                mp["model"],
                mp["params"],
                n_iter=mp["n_iter"],
                random_state=SEMILLA,
                **comunes,
            )
        else:
            busqueda = GridSearchCV(mp["model"], mp["params"], **comunes)
        busqueda.fit(X, y)
        i, r = busqueda.best_index_, busqueda.cv_results_
        filas.append(
            {
                "modelo": nombre,
                "busqueda": "aleatoria" if "n_iter" in mp else "rejilla",
                "combinaciones": len(r["params"]),
                "auc_cv": r["mean_test_auc"][i],
                "auc_std": r["std_test_auc"][i],
                "f1_cv": r["mean_test_f1"][i],
                "auc_train": r["mean_train_auc"][i] if con_train else np.nan,
                "mejores_params": busqueda.best_params_,
                "en_borde": en_borde(mp["params"], busqueda.best_params_),
            }
        )
        ganadores[nombre] = busqueda.best_estimator_
        print(f"{nombre:10s} listo en {time.perf_counter() - t_ini:5.1f} s")
    tabla = pd.DataFrame(filas).sort_values("auc_cv", ascending=False)
    return tabla.reset_index(drop=True), ganadores


tabla, ganadores = tamizar(model_params, X_train, y_train, CV)
n_comb = int(tabla["combinaciones"].sum())
print(f"Combinaciones: {n_comb} | ajustes (x{N_SPLITS} pliegues): {n_comb * N_SPLITS}")
tabla.round(4)

### Mapa de lo explorado (qué se probó)

El documento pide **documentar qué se probó y qué se encontró**. Esta tabla es lo primero (copiarla al README); `mejores_params` y `en_borde` de la tabla anterior son lo segundo.

In [ ]:
espacio_explorado(model_params)

### SVM (kernels linear, poly, rbf y sigmoid)

La SVM con kernel crece de forma cuadrática con el número de filas, así que se tamiza sobre una **submuestra por fechas** (1 de cada 4 días del entrenamiento). Los números **no son directamente comparables** con la tabla anterior (otro tamaño de muestra y otros pliegues); sirven para saber si algún kernel se acerca al resto. La SVM no da probabilidades por defecto, pero `roc_auc` usa su `decision_function`.

In [ ]:
fechas_tr = np.sort(train["fecha"].unique())
sub = train[train["fecha"].isin(fechas_tr[::4])].reset_index(drop=True)
cv_sub = splits_por_fecha(sub["fecha"].to_numpy(), N_SPLITS, GAP // 4 + 1)

svm_espacios = {
    "linear": {"clf__C": [0.1, 1.0, 10.0]},
    "poly": {"clf__C": [0.1, 1.0, 10.0], "clf__degree": [2, 3]},
    "rbf": {"clf__C": [0.1, 1.0, 10.0], "clf__gamma": ["scale", 0.01, 0.1]},
    "sigmoid": {"clf__C": [0.1, 1.0, 10.0], "clf__gamma": ["scale", 0.01, 0.1]},
}
svm_params = {
    f"svm_{kernel}": {
        "model": Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel=kernel))]),
        "params": espacio,
    }
    for kernel, espacio in svm_espacios.items()
}
tabla_svm, ganadores_svm = tamizar(svm_params, sub[COLS_PROPIAS], sub["y"], cv_sub)
print("Filas de la submuestra:", len(sub))
tabla_svm.round(4)

### Poda del árbol (costo-complejidad), como en el Módulo 4

Un árbol sin límite memoriza el entrenamiento: tiene una hoja casi por cada fila. La **poda por costo-complejidad** penaliza cada hoja con un costo `ccp_alpha`: se minimiza `impureza + alpha x (número de hojas)`. Con `alpha = 0` queda el árbol completo; al subir `alpha`, se podan las ramas que menos aportan.

`cost_complexity_pruning_path` calcula los valores de `alpha` donde el árbol realmente cambia. En el Módulo 4 se escogió el `alpha` con la métrica de *test*. Aquí se escoge con la **validación cruzada temporal**. Se reportan dos candidatos:

- **El mejor por AUC en validación** (`alpha_mejor`). Con señal débil suele ser casi un empate entre muchos árboles, y gana uno por ruido.
- **El de un error estándar** (`alpha_1se`): entre los árboles cuyo AUC queda a menos de una desviación del mejor, el **más podado**. Es el que se usa en el resto del notebook: si no se distinguen del mejor, nos quedamos con el más simple.

Los valores de `alpha` se toman con más resolución en los árboles chicos (que es donde está la decisión) y menos en los grandes.

In [ ]:
arbol_completo = DecisionTreeClassifier(min_samples_leaf=25, random_state=SEMILLA)
camino = arbol_completo.cost_complexity_pruning_path(X_train, y_train)
a = camino.ccp_alphas[:-1]  # la última poda deja solo la raíz
pos = len(a) - 1 - np.round(np.geomspace(1, len(a) - 1, 35)).astype(int)
alphas = np.unique(np.r_[0.0, a[pos]])

gs_poda = GridSearchCV(
    arbol_completo,
    {"ccp_alpha": alphas},
    cv=CV,
    scoring=SCORING,
    refit="auc",
    n_jobs=-1,
    return_train_score=True,
)
gs_poda.fit(X_train, y_train)
r = gs_poda.cv_results_
poda = pd.DataFrame(
    {
        "ccp_alpha": np.asarray(r["param_ccp_alpha"], dtype=float),
        "auc_train": r["mean_train_auc"],
        "auc_cv": r["mean_test_auc"],
        "auc_cv_std": r["std_test_auc"],
    }
)
poda["hojas"] = [
    clone(arbol_completo).set_params(ccp_alpha=a).fit(X_train, y_train).get_n_leaves()
    for a in poda["ccp_alpha"]
]

mejor = poda["auc_cv"].idxmax()
limite = poda.loc[mejor, "auc_cv"] - poda.loc[mejor, "auc_cv_std"]
alpha_1se = poda.loc[poda["auc_cv"] >= limite, "ccp_alpha"].max()
alpha_elegido = alpha_1se
alpha_mejor = poda.loc[mejor, "ccp_alpha"]
fila_1se = poda[poda["ccp_alpha"] == alpha_1se].iloc[0]
print(f"Mejor alpha por AUC en CV: {alpha_mejor:.6f}")
print(f"  hojas: {poda.loc[mejor, 'hojas']}")
print(f"Alpha con regla de 1 error estándar: {alpha_1se:.6f}")
print(f"  hojas: {fila_1se['hojas']}")
poda.round(4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
umbral_lin = poda.loc[poda["ccp_alpha"] > 0, "ccp_alpha"].min()
for ax in (ax1, ax2):
    ax.set_xscale("symlog", linthresh=umbral_lin)
    ax.axvline(alpha_mejor, color="green", label="Mejor AUC")
    ax.axvline(alpha_1se, color="red", linestyle="--", label="1 error est. (elegido)")
    ax.set_xlabel("ccp_alpha (más a la derecha = más poda)")

ax1.plot(poda["ccp_alpha"], poda["auc_train"], marker="o", label="Entrenamiento")
ax1.plot(poda["ccp_alpha"], poda["auc_cv"], marker="o", label="Validación cruzada")
ax1.fill_between(
    poda["ccp_alpha"],
    poda["auc_cv"] - poda["auc_cv_std"],
    poda["auc_cv"] + poda["auc_cv_std"],
    alpha=0.2,
)
ax1.axhline(0.5, color="k", linestyle=":", label="Sin información")
ax1.set_title("AUC según la poda del árbol")
ax1.set_ylabel("AUC-ROC")
ax1.legend()

ax2.plot(poda["ccp_alpha"], poda["hojas"], marker="o", color="tab:green")
ax2.set_yscale("log")
ax2.set_title("Tamaño del árbol según la poda")
ax2.set_ylabel("Número de hojas (escala log)")
ax2.legend()
plt.show()

**Cómo leerlo:** a la izquierda (sin poda) el AUC de entrenamiento queda muy por encima del de validación: el árbol memoriza. Al podar, ambos se acercan. Si el AUC de validación casi no cambia (o mejora) al pasar de cientos de hojas a unas pocas, esas ramas extra eran ruido, y el árbol chico es igual de bueno y mucho más fácil de explicar. Un AUC de validación **por debajo de 0.5** no significa que el árbol sirva "al revés": significa que lo aprendido en el pasado no funcionó en el futuro (ruido y cambios de régimen del mercado).

In [ ]:
arbol_podado = clone(arbol_completo).set_params(ccp_alpha=alpha_elegido)
arbol_podado.fit(X_train, y_train)
print("Hojas del árbol completo:", arbol_completo.fit(X_train, y_train).get_n_leaves())
print("Hojas del árbol podado  :", arbol_podado.get_n_leaves())

## 8. Comparación de modelos

Gráfica de comparación (la tercera que pide el documento, junto con la de distribución y la serie de tiempo del notebook 01). La línea punteada es el AUC de un modelo sin información (0.50). Las barras de error son la desviación entre pliegues.

In [ ]:
orden = tabla.sort_values("auc_cv")
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(orden["modelo"], orden["auc_cv"], xerr=orden["auc_std"], color="tab:blue")
ax.axvline(0.5, color="k", linestyle="--", linewidth=1, label="Sin información (0.50)")
ax.set_xlim(0.45, max(0.60, orden["auc_cv"].max() + 0.03))
ax.set_title("AUC-ROC en validación cruzada temporal, por modelo")
ax.set_xlabel("AUC-ROC (promedio de pliegues)")
ax.set_ylabel("Modelo")
ax.legend()
plt.show()

**Cómo leer la tabla.** Un AUC de 0.50 es adivinar. En precios diarios de acciones lo realista es 0.50 a 0.55, así que una diferencia de uno o dos puntos sobre el baseline ya es el resultado, no un fracaso. Si `auc_std` es del tamaño de la ventaja sobre 0.50, esa ventaja no es distinguible del azar. Si algún modelo da 0.70 o más, **sospecha de una fuga de datos** antes de celebrar.

## 9. ¿Ayuda el contexto de SPY y NVDA? (ablación)

Con los tres mejores modelos (sin contar el baseline) se repite la validación con y sin las características de contexto, sobre los mismos pliegues. Los hiperparámetros se dejan fijos (los ganadores de arriba): es una comparación de *entradas*, no una nueva búsqueda.

In [ ]:
top3 = tabla.loc[tabla["modelo"] != "dummy", "modelo"].head(3).tolist()
filas = []
for nombre in top3:
    for etiqueta, X in [("propias", X_train), ("propias + contexto", X_train_ctx)]:
        r = cross_validate(
            clone(ganadores[nombre]), X, y_train, cv=CV, scoring=SCORING, n_jobs=-1
        )
        filas.append(
            {
                "modelo": nombre,
                "features": etiqueta,
                "auc_cv": r["test_auc"].mean(),
                "auc_std": r["test_auc"].std(),
                "f1_cv": r["test_f1"].mean(),
            }
        )
pd.DataFrame(filas).round(4)

Si "propias + contexto" no mejora de forma clara, la conclusión válida es que el contexto no aporta con estos modelos y se deja fuera (más simple, y la app no tiene que descargar SPY y NVDA). Anótalo en la sección de conclusiones.

## 10. Mejor modelo: ROC, umbral de Youden y matriz de confusión

Como en el Módulo 4, se grafica la curva ROC y se elige un umbral con el **estadístico de Youden** (`J = TPR - FPR`, el punto de la curva más lejos de la diagonal). La diferencia clave: las probabilidades vienen de los pliegues de **validación** (predicciones fuera de muestra del entrenamiento), no del holdout.

In [ ]:
def predicciones_fuera_de_muestra(est, X, y, cv):
    idx, prob = [], []
    for tr, va in cv:
        m = clone(est).fit(X.iloc[tr], y.iloc[tr])
        idx.append(va)
        prob.append(m.predict_proba(X.iloc[va])[:, 1])
    idx = np.concatenate(idx)
    return pd.Series(np.concatenate(prob), index=X.index[idx]).sort_index()


ganador = tabla.loc[tabla["modelo"] != "dummy", "modelo"].iloc[0]
print("Modelo ganador por AUC en CV:", ganador)

prob_oof = predicciones_fuera_de_muestra(ganadores[ganador], X_train, y_train, CV)
y_oof = y_train.loc[prob_oof.index]

fpr, tpr, umbrales = roc_curve(y_oof, prob_oof)
umbral = umbrales[np.argmax(tpr - fpr)]
auc_oof = roc_auc_score(y_oof, prob_oof)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, label=f"{ganador} (AUC = {auc_oof:.3f})")
ax.plot([0, 1], [0, 1], "k--", label="Sin información")
ax.scatter(
    fpr[np.argmax(tpr - fpr)],
    tpr[np.argmax(tpr - fpr)],
    color="red",
    zorder=3,
    label=f"Umbral de Youden = {umbral:.3f}",
)
ax.set_title("Curva ROC (validación temporal, fuera de muestra)")
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos")
ax.legend()
plt.show()

In [ ]:
def metricas(y_real, prob, corte):
    pred = (prob >= corte).astype(int)
    return {
        "umbral": round(float(corte), 4),
        "accuracy": accuracy_score(y_real, pred),
        "precision": precision_score(y_real, pred, zero_division=0),
        "recall": recall_score(y_real, pred, zero_division=0),
        "f1": f1_score(y_real, pred, zero_division=0),
    }


pd.DataFrame(
    {
        "corte 0.50": metricas(y_oof, prob_oof, 0.5),
        "corte Youden": metricas(y_oof, prob_oof, umbral),
    }
).T.round(4)

In [ ]:
pred_y = (prob_oof >= umbral).astype(int)
cm = confusion_matrix(y_oof, pred_y)

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")
ax.set_xticks([0, 1], ["Baja", "Sube"])
ax.set_yticks([0, 1], ["Baja", "Sube"])
ax.set_title("Matriz de confusión (umbral de Youden)")
ax.set_xlabel("Predicción")
ax.set_ylabel("Real")
plt.show()

Con un modelo casi aleatorio, el umbral de Youden suele quedar cerca de la proporción de días al alza y la matriz muestra aciertos poco mejores que lanzar una moneda. Eso es normal en este problema; lo importante es medirlo bien y reportarlo con honestidad.

## 11. ¿Dónde sí hay señal? AUC por ticker

Mismas predicciones fuera de muestra, separadas por ticker. Cada ticker tiene pocos días de validación, así que se calcula un margen aproximado de ruido (±2 errores estándar del AUC bajo la hipótesis de que el modelo no sabe nada). Una diferencia menor que ese margen no se puede distinguir del azar. Con 10 tickers, alguno saldrá "bueno" solo por casualidad.

In [ ]:
val = train.loc[prob_oof.index, ["ticker", "y"]].assign(prob=prob_oof)
filas = []
for t, g in val.groupby("ticker"):
    n1 = int(g["y"].sum())
    n0 = len(g) - n1
    se = np.sqrt((len(g) + 1) / (12 * n1 * n0))
    filas.append(
        {
            "ticker": t,
            "n_dias": len(g),
            "auc": roc_auc_score(g["y"], g["prob"]),
            "ruido_2se": 2 * se,
        }
    )
auc_por_ticker = pd.DataFrame(filas).sort_values("auc", ascending=False)
auc_por_ticker.round(4)

## 12. Importancia de variables

Importancia por **permutación** sobre el último pliegue de validación: se desordena una columna a la vez y se mide cuánto cae el AUC. Funciona con cualquier modelo (en el Módulo 4 se usaron los coeficientes y `feature_importances_`, que dependen del tipo de modelo). Valores cercanos a cero significan que esa variable no ayuda.

In [ ]:
tr_u, va_u = CV[-1]
m_imp = clone(ganadores[ganador]).fit(X_train.iloc[tr_u], y_train.iloc[tr_u])
imp = permutation_importance(
    m_imp,
    X_train.iloc[va_u],
    y_train.iloc[va_u],
    scoring="roc_auc",
    n_repeats=10,
    random_state=SEMILLA,
    n_jobs=-1,
)
serie = pd.Series(imp.importances_mean, index=COLS_PROPIAS).sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(serie.index, serie.values, color="tab:blue")
ax.axvline(0, color="k", linewidth=1)
ax.set_title(f"Importancia por permutación ({ganador}, último pliegue)")
ax.set_xlabel("Caída del AUC al desordenar la variable")
ax.set_ylabel("Variable")
plt.show()

## 13. Árbol interpretable (como en el Módulo 4)

Aunque el árbol no sea el ganador, graficarlo ayuda a explicar con qué variables parte los datos. Se grafica el **árbol podado** de la sección 7 y solo sus primeros 3 niveles para que se lea.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(
    arbol_podado,
    feature_names=COLS_PROPIAS,
    class_names=["Baja", "Sube"],
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3,
    ax=ax,
)
ax.set_title("Árbol podado (se muestran los 3 primeros niveles)")
plt.show()

pd.Series(arbol_podado.feature_importances_, index=COLS_PROPIAS).sort_values(
    ascending=False
).head(8).round(4)

## 14. Holdout (una sola vez, al final)

Esta celda está **apagada** (`EVALUAR_HOLDOUT = False`). Se enciende cuando ya no vayas a cambiar nada: modelo, características y hiperparámetros congelados. Si miras el holdout, ajustas algo y vuelves a mirar, el holdout deja de ser independiente. La evaluación definitiva se hará con el pipeline de `src/`; esta celda es solo un vistazo.

In [ ]:
EVALUAR_HOLDOUT = False

if EVALUAR_HOLDOUT:
    final = clone(ganadores[ganador]).fit(X_train, y_train)
    prob_h = final.predict_proba(hold[COLS_PROPIAS])[:, 1]
    print("AUC holdout:", round(roc_auc_score(hold["y"], prob_h), 4))
    print(pd.Series(metricas(hold["y"], prob_h, umbral)).round(4))

## 15. Conclusiones

- **Baseline (`dummy`, siempre "sube"):** AUC 0.500 y F1 0.692. El F1 del dummy es alto porque el 52.7 % de los días suben; por eso el AUC es la métrica principal.
- **Mejor modelo por AUC en CV:** AdaBoost con AUC 0.5097 ± 0.013. La regresión logística (0.5093), el bagging (0.5091) y el árbol (0.5090) quedan a menos de 0.001: es un **empate**, no un ganador.
- **¿Supera de forma clara al baseline?** No. Todos los modelos clásicos quedan entre 0.506 y 0.510 y la desviación entre pliegues es de 0.013 o más, mucho mayor que la diferencia. El SVM con kernel RBF llegó a 0.5235, pero se entrenó con una submuestra de 7,000 filas y varía 0.017 entre pliegues, así que no se toma como mejora real.
- **¿Ayudó el contexto SPY/NVDA?** No, empeoró: AdaBoost 0.5097 a 0.5013, logística 0.5093 a 0.4998 y bagging 0.5091 a 0.5023. Se descarta.
- **Qué usó el modelo:** casi todo el peso lo llevan `desv_5d` (60 %) y `ret_1d` (40 %): la volatilidad reciente y el retorno de ayer.
- **Por acción:** el AUC va de 0.484 (ANF) a 0.526 (IBM), pero el margen de ruido es de ±0.024 y el orden cambió al mover la fecha de corte unos días, así que no hay acciones "más predecibles".
- **Sobreajuste visible:** el boosting llega a 0.82 de AUC en entrenamiento y 0.508 en validación. El árbol completo (852 hojas) tiene AUC de CV 0.496; podado a 4 hojas con la regla de 1 error estándar rinde igual, así que la poda simplifica sin costo.
- **Dos modelos finalistas (pipeline en `src/`, MLflow):** la **regresión logística** (probabilidades suaves, coeficientes interpretables y empata con el mejor) y el **MLP de PyTorch** (notebook 03).
- **Limitaciones para el README:** sesgo de supervivencia (tickers elegidos hoy), AUC cercano a 0.5 como resultado esperable para dirección diaria, y los precios y el volumen dependen de Yahoo Finance. El holdout (30 de mar a 30 de sep de 2026) sigue sin abrirse.

**Qué se mueve a `src/` después:** `features_propias` y `features_contexto` (módulo `features`), `splits_por_fecha` (módulo `evaluation`), `tamizar` y los `model_params` (módulo `models`), todo leyendo `configs/config.yaml`.